# 04 · Gradients and learning / Gradient နှင့် QML အခြေခံ

**Goal:** differentiate an expectation and train a small circuit. About 20 minutes.
Parameter-shift changes one gate occurrence at a time and applies the arithmetic chain rule.
This SDK API is explicit optimization, not PyTorch autograd.

Parameter တစ်ခုကို gate များစွာတွင် ပြန်သုံးထားလျှင် derivative များကို chain rule ဖြင့် ပေါင်းပါသည်။


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nexqus
from nexqus import Circuit, Simulator, Parameter, Noise, NoiseModel, algorithms
assert nexqus.__version__.split('.')[0] == '1', 'Install NexQus SDK 1.x in this kernel environment'
print('NexQus SDK:', nexqus.__version__)


In [ ]:
from nexqus import QuantumFunction, QuantumRegressor, QuantumBinaryClassifier, Adam
theta = Parameter('theta')
function = QuantumFunction(Circuit(1).ry(theta, 0).ry(2*theta, 0), 'Z')
value, gradients = function.value_and_grad({'theta': .2})
assert np.isclose(value, np.cos(.6))
assert np.isclose(gradients['theta'], -3*np.sin(.6))
h = 1e-6
finite_difference = (function({'theta': .2+h}) - function({'theta': .2-h}))/(2*h)
np.testing.assert_allclose(gradients['theta'], finite_difference, atol=1e-8)
print('Analytic gradient:', gradients['theta'])
print('Finite-difference check:', finite_difference)


In [ ]:
x, weight, bias = Parameter('x'), Parameter('weight'), Parameter('bias')
template = Circuit(1).ry(weight*x + bias, 0)
function = QuantumFunction(template, 'Z')
model = QuantumRegressor(function, ['x'], {'weight': .7, 'bias': -.8})
train_x = np.linspace(-np.pi, np.pi, 20); train_y = np.sin(train_x)
initial_loss = float(np.mean((model.predict(train_x) - train_y)**2))
# Adam ဖြင့် weight နှင့် bias ကို train ပါ။
model.fit(train_x, train_y, epochs=80, optimizer=Adam(lr=.05))
grid = np.linspace(-np.pi, np.pi, 101); predictions = model.predict(grid)
grid_mse = float(np.mean((predictions-np.sin(grid))**2))
print('Initial training MSE:', initial_loss)
print('Final evaluation-grid MSE:', grid_mse)
print('Learned parameters:', model.parameters)
assert grid_mse < 1e-4


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].semilogy([r['loss'] for r in model.history]); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE before update'); axes[0].set_title('Training history')
axes[1].plot(grid, np.sin(grid), label='Target sin(x)')
axes[1].plot(grid, predictions, '--', label='Circuit prediction')
axes[1].set_xlabel('x'); axes[1].set_title('Dense evaluation grid'); axes[1].legend()
plt.tight_layout(); plt.show()


The circuit computes cos(weight*x+bias). Sine is representable at weight=1, bias=-pi/2.
This is a learning demonstration, not evidence of quantum advantage or performance on real datasets.
Pauli expectations lie in [-1,1], so the built-in MSE model requires scaled targets.

ဤ example သည် synthetic data ပေါ်တွင် circuit parameter သင်ယူပုံကို ပြသခြင်းဖြစ်ပါသည်။
Classical model ထက် ပိုကောင်းကြောင်း သက်သေမဟုတ်ပါ။


In [ ]:
classifier = QuantumBinaryClassifier(function, ['x'], {'weight': .7, 'bias': -1.0})
features = np.array([-1.2, -.7, -.3, .3, .7, 1.2]); labels = (features > 0).astype(int)
classifier.fit(features, labels, epochs=35, optimizer=Adam(lr=.06))
np.testing.assert_array_equal(classifier.predict(features), labels)
print('Toy classification labels:', classifier.predict(features))
print('Scores:', classifier.predict_proba(features))


**Limits:** binary scores are not guaranteed calibrated probabilities. Trajectory gradients, dynamic-circuit
gradients, multiclass learning and a Torch neural-network layer are not included.
**Try it:** vary the initial parameters or training rate and compare the recorded losses.
